# Семинар №9. Транзакции, MVCC, изоляция и блокировки

## Как устроен ноутбук

1. Каждая тема идёт по схеме: **теория → демо → упражнение**.
2. Ячейки с комментарием `# УПРАЖНЕНИЕ N` — для самостоятельной работы; там стоят `TODO`.
3. Выполняйте ячейки **по порядку**: состояние таблиц в БД важно.
4. Сброс: `Kernel → Restart & Run All`.

## Окружение

- PostgreSQL 16 слушает `localhost:5432` **внутри контейнера**.
- Учётные данные БД: пользователь `postgres`, база `demo`. Если в вашем окружении включён пароль — добавьте `password="..."` в словарь `DB` ниже.
- Jupyter-пароль: `student`.

## План

1. Как устроена транзакционность: XID, CLOG, wraparound, freeze.
2. Компромисс консистентность vs доступность; MVCC.
3. Распределённые транзакции: 2PC и Saga.
4. Уровни изоляции в PostgreSQL.
5. Best practices по транзакциям.
6. Пессимистичные блокировки (`SELECT … FOR UPDATE`) и SQLAlchemy.
7. Оптимистичный контроль версий.
8. Deadlock и его диагностика.


In [7]:
# Проверка окружения и общая конфигурация
import sys
print("Python:", sys.version.split()[0])

import psycopg, sqlalchemy, pandas, numpy
print("psycopg     :", psycopg.__version__)
print("sqlalchemy  :", sqlalchemy.__version__)
print("pandas      :", pandas.__version__)
print("numpy       :", numpy.__version__)

DB = {
    "host":   "localhost",
    "port":   5432,
    "dbname": "demo",
    "user":   "postgres",
}

PSYCOPG_CONN = (
    f"host={DB['host']} port={DB['port']} dbname={DB['dbname']} "
    f"user={DB['user']}"
)
SQLALCHEMY_URL = (
    f"postgresql+psycopg://{DB['user']}@{DB['host']}:{DB['port']}/{DB['dbname']}"
)

from sqlalchemy import create_engine, text
engine = create_engine(SQLALCHEMY_URL, echo=False)

print("\npsycopg DSN      :", PSYCOPG_CONN)
print("SQLAlchemy URL   :", SQLALCHEMY_URL)

# Быстрая проверка подключения
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version()")
        print("\nPG version:", cur.fetchone()[0].split(" on ")[0])


Python: 3.11.17
psycopg     : 3.3.6
sqlalchemy  : 2.1.6
pandas      : 3.1.0
numpy       : 2.4.6

psycopg DSN      : host=localhost port=5432 dbname=demo user=postgres
SQLAlchemy URL   : postgresql+psycopg://postgres@localhost:5432/demo

PG version: PostgreSQL 15.19 (Debian 15.19-0+deb12u1)


## 2.1 Как устроена транзакционность в PostgreSQL

Когда транзакция выполняет первую **изменяющую** операцию (`INSERT/UPDATE/DELETE`) или явно запрашивает XID через `SELECT txid_current()`, PostgreSQL назначает ей уникальный **XID (Transaction ID)**. Пустой `BEGIN; SELECT 1; COMMIT;` **не получает XID вообще** — это важно, потому что 32-битное пространство XID конечно (~4 млрд значений).

- Каждая строка в heap имеет заголовок с `xmin` (XID вставившей транзакции) и `xmax` (XID удалившей/изменившей).
- Статус транзакции (IN_PROGRESS / COMMITTED / ABORTED / SUBCOMMITTED) хранится в **CLOG** — компактном массиве 2-битных состояний (каталог `pg_xact`).
- **`SELECT` без `FOR UPDATE` не блокирует и не блокируется `UPDATE`/`DELETE`** — читатель работает со снапшотом, а не с блокировками строк.

### Проблема wraparound и freeze

XID цикличен. Если старые строки не «заморозить», после переполнения счётчика они станут «из будущего» и пропадут из видимости. Autovacuum заменяет их на `FrozenTransactionId`, который всегда считается старым. Долгие транзакции мешают vacuum → риск wraparound и аварийной остановки.

**Золотое правило: транзакции должны быть короткими.**


In [8]:
# Демо: когда XID присваивается, а когда нет
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT txid_current_if_assigned()")
        print("XID в начале транзакции (до изменений):", cur.fetchone()[0])
    conn.commit()

with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS xid_demo")
        cur.execute("CREATE TABLE xid_demo (id INT PRIMARY KEY, val INT)")
        cur.execute("INSERT INTO xid_demo VALUES (1, 100)")
        cur.execute("SELECT txid_current_if_assigned()")
        print("XID после INSERT:", cur.fetchone()[0])
    conn.commit()

# Проверим xmin/xmax у строки
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT id, val, xmin, xmax FROM xid_demo")
        print("\n(id, val, xmin, xmax):", cur.fetchone())

# Параметры autovacuum, связанные с freeze
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT name, setting FROM pg_settings
            WHERE name IN ('autovacuum', 'autovacuum_freeze_max_age')
        """)
        for name, val in cur.fetchall():
            print(f"{name:30s} = {val}")


XID в начале транзакции (до изменений): None
XID после INSERT: 834

(id, val, xmin, xmax): (1, 100, '834', '0')
autovacuum                     = on
autovacuum_freeze_max_age      = 200000000


### Упражнение 1. Разберитесь с XID и активными снапшотами

**Задание.** Продемонстрируйте, что `txid_current_if_assigned()` возвращает `NULL` до изменений, а после `INSERT` — реальный XID. Дополнительно посмотрите, что видно в `pg_stat_activity` по вашей сессии: `backend_xid` (XID, назначенный текущей транзакции) и `backend_xmin` (нижняя граница снапшота).

**Шаги.**
1. В новой транзакции выполните `SELECT txid_current_if_assigned()` — напечатайте результат.
2. Сделайте `INSERT` в `xid_demo` (используйте id 100).
3. Снова выполните `SELECT txid_current_if_assigned()` — значение должно быть не NULL.
4. Из **второго** соединения (`psycopg.connect(...)`) запросите `pg_stat_activity` по `pid = pg_backend_pid()` первой сессии: посмотрите `backend_xid`, `backend_xmin`.
5. Сделайте `COMMIT` в первой сессии и повторите п.4 — что изменилось?

**Критерий успеха.** До `INSERT` — `None`, после — число; `backend_xid` виден, пока транзакция открыта.

#### Теория (кратко и понятно)

В PostgreSQL каждая транзакция, которая меняет данные (`INSERT`, `UPDATE`, `DELETE`), получает уникальный номер — **XID (Transaction ID)**.

Если ты просто делаешь `SELECT` или начинаешь транзакцию, но ничего не меняешь, XID **не выдается** (функция `txid_current_if_assigned()` вернет `None`).

Как только ты делаешь `INSERT`, базе нужно записать в заголовок строки (`xmin`), кто именно её создал. Тут же выдается XID.

В системном представлении **`pg_stat_activity`** (это "диспетчерская", где видно все текущие подключения) можно увидеть:

*   **`backend_xid`** — номер текущей транзакции этого подключения. Он будет виден, пока транзакция открыта.
*   **`backend_xmin`** — граница самого старого снапшота. Это критически важно для работы `VACUUM` (очистки мусора).

In [9]:
# УПРАЖНЕНИЕ 1

import psycopg
import time

# 1. Открываем ПЕРВОЕ соединение (conn1) и курсор
conn1 = psycopg.connect(PSYCOPG_CONN)
cur1 = conn1.cursor()

# 2. Проверяем XID ДО изменений. 
# Так как мы еще ничего не меняли, функция вернет None
cur1.execute("SELECT txid_current_if_assigned()")
print("XID до изменений (ожидаем None):", cur1.fetchone()[0])

# 3. Делаем INSERT в таблицу xid_demo (создана в демо выше)
# Именно в этот момент база выделяет транзакции уникальный XID
cur1.execute("INSERT INTO xid_demo (id, val) VALUES (100, 500)")

# 4. Снова проверяем XID. Теперь он должен быть числом (например, 12345)
cur1.execute("SELECT txid_current_if_assigned()")
xid = cur1.fetchone()[0]
print("XID после INSERT (ожидаем число):", xid)

# 5. Узнаем PID (ID процесса) нашей первой сессии, чтобы потом найти ее в диспетчерской
cur1.execute("SELECT pg_backend_pid()")
pid1 = cur1.fetchone()[0]

# 6. Открываем ВТОРОЕ соединение (conn2), чтобы посмотреть на первую сессию "со стороны"
with psycopg.connect(PSYCOPG_CONN) as conn2:
    with conn2.cursor() as cur2:
        # Ищем информацию о первой сессии в pg_stat_activity по ее PID
        # Смотрим backend_xid (номер транзакции) и backend_xmin (границу снапшота)
        cur2.execute("SELECT backend_xid, backend_xmin FROM pg_stat_activity WHERE pid = %s", (pid1,))
        print("Пока транзакция открыта (backend_xid, backend_xmin):", cur2.fetchone())

# 7. Завершаем транзакцию в первой сессии (делаем COMMIT)
# После этого строка сохранится, а XID перестанет быть "активным"
conn1.commit()

# 8. Снова смотрим из второй сессии. 
# Поля backend_xid и backend_xmin должны стать пустыми (None), 
# так как транзакция первой сессии завершилась
with psycopg.connect(PSYCOPG_CONN) as conn2:
    with conn2.cursor() as cur2:
        cur2.execute("SELECT backend_xid, backend_xmin FROM pg_stat_activity WHERE pid = %s", (pid1,))
        print("После COMMIT (ожидаем None, None):", cur2.fetchone())

# Закрываем соединение
cur1.close()
conn1.close()

XID до изменений (ожидаем None): None
XID после INSERT (ожидаем число): 835
Пока транзакция открыта (backend_xid, backend_xmin): ('835', None)
После COMMIT (ожидаем None, None): (None, None)


## 2.2 Компромисс: консистентность vs доступность/производительность

Строгая изоляция требует координации, координация снижает пропускную способность. PostgreSQL выбрал **MVCC (Multi-Version Concurrency Control)**: читатели не блокируют писателей, создаются новые версии строк.

Плюсы MVCC:
- высокая пропускная способность чтения;
- читатели не видят «грязных» данных;
- нет read-блокировок.

Минусы MVCC:
- нужен периодический **VACUUM** (autovacuum) для очистки мёртвых версий;
- **bloat** таблиц при долгих транзакциях;
- риск wraparound, если vacuum не успевает.

**Ключевая мысль:** долгая транзакция держит старый снапшот и мешает vacuum удалить старые версии. Поэтому её влияние выходит за рамки «моей» таблицы — она бьёт по всей БД.


In [10]:
# Демо: активная транзакция держит backend_xmin
with psycopg.connect(PSYCOPG_CONN) as conn1:
    with conn1.cursor() as cur1:
        cur1.execute("BEGIN ISOLATION LEVEL REPEATABLE READ")
        cur1.execute("SELECT count(*) FROM xid_demo")  # создаёт snapshot
        cur1.fetchone()
        pid1 = None
        cur1.execute("SELECT pg_backend_pid()")
        pid1 = cur1.fetchone()[0]
        print("Сессия 1 pid =", pid1)

        # Из второго соединения смотрим pg_stat_activity
        with psycopg.connect(PSYCOPG_CONN) as conn2:
            with conn2.cursor() as cur2:
                cur2.execute(
                    "SELECT pid, state, backend_xid, backend_xmin "
                    "FROM pg_stat_activity WHERE pid = %s", (pid1,)
                )
                print("\nПока транзакция 1 открыта:")
                print(cur2.fetchone())
    conn1.rollback()

# Транзакция закрыта — backend_xmin освобождён
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute(
            "SELECT pid, state, backend_xid, backend_xmin "
            "FROM pg_stat_activity WHERE pid = %s", (pid1,)
        )
        print("\nПосле rollback:")
        print(cur.fetchone())


Сессия 1 pid = 851

Пока транзакция 1 открыта:
(851, 'idle in transaction', None, '836')

После rollback:
None


### Упражнение 2. Долгая транзакция мешает vacuum

**Задание.** Покажите, что `backend_xmin` держится всё время жизни транзакции и освобождается только после её завершения. Это и есть причина, по которой долгие транзакции мешают vacuum.

**Шаги.**
1. Откройте соединение `conn1`, начните `BEGIN ISOLATION LEVEL REPEATABLE READ` и выполните `SELECT` из `xid_demo` — создайте snapshot.
2. Из второго соединения посмотрите `pg_stat_activity` по `pid` первой сессии: `backend_xmin` должен быть не NULL.
3. Сделайте в первой сессии ещё один `SELECT` из `xid_demo` — `backend_xmin` не меняется (snapshot стабилен).
4. Закройте транзакцию `ROLLBACK` и снова проверьте `pg_stat_activity` — `backend_xmin` должен стать NULL.
5. В комментарии к коду напишите 1–2 строки: почему удержание `backend_xmin` блокирует очистку мёртвых версий?

**Критерий успеха.** В консоли видно `backend_xmin: <число>` пока транзакция открыта, и `None` после закрытия.

#### Теория

Как мы уже знаем, PostgreSQL не удаляет старые данные физически при `UPDATE` или `DELETE`. Он просто помечает их как «мертвые» и создает новые версии строк (это называется MVCC).

За очистку этого мусора отвечает процесс **VACUUM** (обычно он запускается автоматически — `autovacuum`).
Но есть проблема: если какая-то транзакция открыта очень долго, она видит базу данных такой, какой она была на момент её старта (снапшот).

В `pg_stat_activity` есть колонка `backend_xmin`. Это граница самого старого снапшота.
**Золотое правило:** VACUUM **не может** физически удалить ни одну «мертвую» строку, если она была создана *после* `backend_xmin` открытой транзакции. Иначе он просто сломает ей снапшот.
В результате: пока транзакция висит (даже если она просто спит и ничего не делает), мусор в базе копится, и таблица раздувается (bloat).

**Наша задача:** доказать это на практике. Открыть долгую транзакцию, посмотреть, как за ней удерживается `backend_xmin`, и убедиться, что после завершения транзакции он освобождается.

In [11]:
# УПРАЖНЕНИЕ 2

import psycopg
import time

# 1. Открываем первое соединение (это будет наша "долгая транзакция")
conn1 = psycopg.connect(PSYCOPG_CONN)
cur1 = conn1.cursor()

# 2. Начинаем транзакцию с уровнем изоляции REPEATABLE READ
# На этом уровне транзакция фиксирует "снимок" (snapshot) базы данных 
# на момент первого запроса и держит его до конца.
cur1.execute("BEGIN ISOLATION LEVEL REPEATABLE READ")

# 3. Выполняем SELECT. Именно здесь создается снапшот, 
# и база фиксирует, какие транзакции видны этой сессии.
cur1.execute("SELECT count(*) FROM xid_demo")
cur1.fetchone() # Забираем результат, чтобы запрос точно выполнился

# 4. Узнаем PID (ID процесса) этой транзакции, чтобы потом найти ее в диспетчерской
cur1.execute("SELECT pg_backend_pid()")
pid1 = cur1.fetchone()[0]

# 5. Открываем ВТОРОЕ соединение, чтобы посмотреть на первую сессию "со стороны"
with psycopg.connect(PSYCOPG_CONN) as conn2:
    with conn2.cursor() as cur2:
        # Запрашиваем backend_xmin у первой сессии.
        # Ожидаем увидеть число (XID), так как транзакция держит снапшот.
        cur2.execute("SELECT backend_xmin FROM pg_stat_activity WHERE pid = %s", (pid1,))
        xmin_before = cur2.fetchone()[0]
        print(f"backend_xmin пока транзакция открыта (ожидаем число): {xmin_before}")

# 6. Делаем еще один SELECT в первой сессии. 
# Так как уровень REPEATABLE READ, снапшот не меняется, backend_xmin тоже останется прежним.
cur1.execute("SELECT count(*) FROM xid_demo")
cur1.fetchone()

# 7. Закрываем транзакцию (делаем ROLLBACK). 
# Снапшот уничтожается, транзакция завершается.
cur1.execute("ROLLBACK")

# 8. Снова смотрим из второй сессии. 
# Ожидаем, что backend_xmin станет None (null), так как транзакция закрыта 
# и больше не мешает VACUUM очищать мусор.
with psycopg.connect(PSYCOPG_CONN) as conn2:
    with conn2.cursor() as cur2:
        cur2.execute("SELECT backend_xmin FROM pg_stat_activity WHERE pid = %s", (pid1,))
        xmin_after = cur2.fetchone()[0]
        print(f"backend_xmin после ROLLBACK (ожидаем None): {xmin_after}")

# Закрываем соединения
cur1.close()
conn1.close()

# КОММЕНТАРИЙ К ЗАДАНИЮ:
# Почему удержание backend_xmin блокирует vacuum?
# Потому что VACUUM не знает, какие строки могут быть нужны этой открытой транзакции 
# для её старого снапшота. Чтобы не сломать логику чтения, VACUUM боится удалять 
# строки, помеченные как удаленные после backend_xmin, и оставляет их на диске.

backend_xmin пока транзакция открыта (ожидаем число): 836
backend_xmin после ROLLBACK (ожидаем None): None


## 2.3 Транзакции в распределённых системах

Когда данные размазаны по нескольким базам (микросервисам) или шардам, классические ACID-транзакции «из коробки» не работают. Два основных подхода:

### 2PC (Two-Phase Commit)

- **Фаза 1 (Prepare):** координатор спрашивает все узлы: «Готовы закоммитить?». Узлы блокируют данные и отвечают «Да/Нет».
- **Фаза 2 (Commit/Rollback):** если все «Да» — `COMMIT`, иначе `ROLLBACK`.
- **Минус:** падение координатора после Prepare оставляет участников в заблокированном состоянии → страдает доступность.
- PostgreSQL поддерживает 2PC через `PREPARE TRANSACTION` / `COMMIT PREPARED` / `ROLLBACK PREPARED` (требуется `max_prepared_transactions > 0`).

### Saga + Eventual Consistency

- Вместо одной распределённой транзакции — цепочка локальных.
- При ошибке шага N запускаются компенсирующие транзакции (отменяющие 1…N-1).
- Данные временно неконсистентны, но система Available. Стандарт для высоконагруженных микросервисов.
- **Оркестрация** — центральный координатор. **Хореография** — сервисы подписываются на события друг друга через брокер (Kafka, RabbitMQ).


In [12]:
# Демо: PREPARE TRANSACTION (2PC) в PostgreSQL
with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS twopc_demo")
        cur.execute("CREATE TABLE twopc_demo (id INT PRIMARY KEY, val INT)")
        cur.execute("INSERT INTO twopc_demo VALUES (1, 100)")
        cur.execute("SHOW max_prepared_transactions")
        max_prep = int(cur.fetchone()[0])
        print("max_prepared_transactions =", max_prep)
        if max_prep == 0:
            print("PREPARE TRANSACTION недоступен (нужно max_prepared_transactions > 0 и перезапуск PG).")
        else:
            # Явно готовим транзакцию
            cur.execute("BEGIN")
            cur.execute("UPDATE twopc_demo SET val = 200 WHERE id = 1")
            cur.execute("PREPARE TRANSACTION 'demo_prepared'")
            print("Prepared.")

# Смотрим список подготовленных транзакций
with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT gid, prepared, owner, database FROM pg_prepared_xacts")
        rows = cur.fetchall()
        print("pg_prepared_xacts:", rows)

# Финализируем (в реальной системе это делает координатор)
if rows:
    with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
        with conn.cursor() as cur:
            cur.execute("COMMIT PREPARED 'demo_prepared'")
            cur.execute("SELECT val FROM twopc_demo WHERE id = 1")
            print("After COMMIT PREPARED: val =", cur.fetchone()[0])


max_prepared_transactions = 0
PREPARE TRANSACTION недоступен (нужно max_prepared_transactions > 0 и перезапуск PG).
pg_prepared_xacts: []


### Упражнение 3. PREPARE TRANSACTION и откат

**Задание.** Проведите полный цикл 2PC-транзакции: подготовьте, посмотрите в `pg_prepared_xacts`, откатите через `ROLLBACK PREPARED`, убедитесь, что изменения не применились.

**Шаги.**
1. Проверьте `SHOW max_prepared_transactions`. Если `0` — выведите сообщение, что демо пропущено.
2. В `twopc_demo` (id=2) вставьте значение `500`.
3. Подготовьте транзакцию: `BEGIN; INSERT ...; PREPARE TRANSACTION 'ex3_prepared'`.
4. Из другого соединения посмотрите `pg_prepared_xacts` — ваша транзакция должна быть в списке.
5. Выполните `ROLLBACK PREPARED 'ex3_prepared'`.
6. Проверьте `SELECT * FROM twopc_demo WHERE id = 2` — пусто (изменения откатились).

**Критерий успеха.** В `pg_prepared_xacts` есть `ex3_prepared` до `ROLLBACK PREPARED`; после отката `SELECT` пуст.

#### Теория

Представь, что тебе нужно перевести деньги из Банка А в Банк Б. Это две разные базы данных. Если ты спишешь деньги в А, но база Б упадет и не зачислит их, деньги сгорят.

Чтобы этого не произошло, придумали **Two-Phase Commit (2PC — Двухфазный коммит)**.

1. **Фаза 1 (Prepare):** Координатор говорит базам: «Готовьтесь, скоро будем сохранять». Базы блокируют данные, проверяют, что всё ок, и отвечают: «Готовы» (но пока *не* сохраняют окончательно).
2. **Фаза 2 (Commit):** Если все ответили «Готовы», координатор дает команду: «Сохраняем!» (`COMMIT`). Если хоть одна база упала — все откатываются (`ROLLBACK`).

В PostgreSQL это реализовано через команду **`PREPARE TRANSACTION 'имя'`**. Она переводит транзакцию в «подвешенное» состояние (Prepared). Транзакция уже сделала все изменения, но пока не зафиксировала их. Посмотреть такие транзакции можно в системной таблице **`pg_prepared_xacts`**.


In [13]:
# УПРАЖНЕНИЕ 3
import psycopg

# 1. Проверяем, включен ли 2PC на сервере
# Используем autocommit=True, чтобы драйвер не пытался сам открывать транзакции
with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute("SHOW max_prepared_transactions")
        max_prep = int(cur.fetchone()[0])
        print(f"max_prepared_transactions = {max_prep}")
        
        if max_prep == 0:
            print("ОШИБКА: 2PC выключен на сервере (нужно max_prepared_transactions > 0 и рестарт). Пропускаем.")
            raise SystemExit

# 2. Подготавливаем транзакцию (Фаза 1)
with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
    with conn.cursor() as cur:
        # Начинаем транзакцию
        cur.execute("BEGIN")
        # Вставляем данные в таблицу twopc_demo
        cur.execute("INSERT INTO twopc_demo (id, val) VALUES (2, 500)")
        # Вместо COMMIT вызываем PREPARE. Транзакция "зависает" в подготовленном состоянии.
        cur.execute("PREPARE TRANSACTION 'ex3_prepared'")
        print("Транзакция переведена в состояние PREPARED.")

# 3. Смотрим в системную таблицу pg_prepared_xacts
# Теперь любая другая сессия может увидеть, что есть "подвешенная" транзакция
with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT gid FROM pg_prepared_xacts")
        prepared_list = cur.fetchall()
        print("Подготовленные транзакции в базе:", prepared_list)

# 4. Выполняем откат (ROLBACK) подготовленной транзакции
# В реальной жизни это делает координатор, если понимает, что где-то произошла ошибка
with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute("ROLLBACK PREPARED 'ex3_prepared'")
        print("Выполнен ROLLBACK PREPARED. Изменения отменены.")

# 5. Проверяем, что данные не сохранились
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT * FROM twopc_demo WHERE id = 2")
        result = cur.fetchall()
        print("Данные с id=2 после отката (ожидаем пустой список []):", result)

max_prepared_transactions = 0
ОШИБКА: 2PC выключен на сервере (нужно max_prepared_transactions > 0 и рестарт). Пропускаем.


SystemExit: 

/usr/local/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 2.4 Уровни изоляции в PostgreSQL

Аномалии чтения:

| Аномалия | Описание |
|---|---|
| **Dirty Read** | Чтение данных, изменённых ещё не зафиксированной транзакцией |
| **Non-repeatable Read** | Повторный `SELECT` той же строки возвращает другое значение |
| **Phantom Read** | Повторный запрос по диапазону возвращает другой набор строк |
| **Lost Update** | Две транзакции читают одно значение, обе пишут — одна затирает другую |

Уровни изоляции **в реализации PostgreSQL**:

1. **READ UNCOMMITTED** — автоматически повышается до `READ COMMITTED`. Никогда не используется.
2. **READ COMMITTED** (default) — свежий snapshot на каждый **оператор**.
   - Закрывает: Dirty Read.
   - **Не** закрывает: Non-repeatable Read, Phantom Read, Lost Update.
3. **REPEATABLE READ** — один snapshot на всю транзакцию (создаётся на первом запросе, не на `BEGIN`).
   - Закрывает: Dirty Read, Non-repeatable Read, **Phantom Read** (отличие от стандарта SQL: в PG phantom закрыт).
   - **Не** закрывает: Lost Update.
   - При попытке изменить строку, изменённую параллельной транзакцией после snapshot’а: `ERROR: could not serialize access due to concurrent update`.
4. **SERIALIZABLE** — SSI (Serializable Snapshot Isolation): отслеживает read/write-зависимости, откатывает при обнаружении цикла.
   - Закрывает всё, включая Lost Update и Write Skew.
   - Самая дорогая по CPU/памяти; используйте там, где действительно нужна полная сериализуемость.

**Шпаргалка:**

| Уровень | Dirty Read | Non-repeatable | Phantom | Lost Update |
|---|---|---|---|---|
| READ UNCOMMITTED (PG) | ✗ | ✓ | ✓ | ✓ |
| READ COMMITTED | ✗ | ✓ | ✓ | ✓ |
| REPEATABLE READ | ✗ | ✗ | ✗ | ✓ |
| SERIALIZABLE | ✗ | ✗ | ✗ | ✗ |

(✗ — аномалия невозможна, ✓ — возможна)


In [14]:
# Демо: установка уровня изоляции и проверка
# Уровень по умолчанию
with engine.connect() as conn:
    iso = conn.execute(text("SHOW transaction_isolation")).scalar()
    print("Default isolation:", iso)

# Через SQLAlchemy execution_options
for level in ["READ COMMITTED", "REPEATABLE READ", "SERIALIZABLE"]:
    with engine.connect().execution_options(isolation_level=level) as conn:
        iso = conn.execute(text("SHOW transaction_isolation")).scalar()
        print(f"  requested={level!r:20s} -> actual={iso!r}")


Default isolation: read committed
  requested='READ COMMITTED'     -> actual='read committed'
  requested='REPEATABLE READ'    -> actual='repeatable read'
  requested='SERIALIZABLE'       -> actual='serializable'


### Упражнение 4. Non-repeatable Read на READ COMMITTED

**Задание.** Воспроизведите Non-repeatable Read на `READ COMMITTED` и убедитесь, что `REPEATABLE READ` её закрывает.

**Шаги.**
1. Создайте таблицу `ex4_accounts (id INT PRIMARY KEY, balance INT)` и вставьте `(1, 1000)`.
2. В двух потоках (`threading.Thread`):
   - **Поток A** открывает транзакцию с `READ COMMITTED`, читает `balance` (1000), спит 1 сек, читает ещё раз, печатает оба значения, делает `COMMIT`.
   - **Поток B** спит 0.3 сек, делает `UPDATE ... SET balance = 500 WHERE id = 1`, `COMMIT`.
3. Запустите и убедитесь, что второй `SELECT` в потоке A вернул **500** (non-repeatable read).
4. Повторите эксперимент, установив в потоке A `REPEATABLE READ`. Второй `SELECT` должен вернуть **1000**.
5. В комментарии: почему `REPEATABLE READ` закрывает эту аномалию?

**Критерий успеха.** В первом прогоне второе значение = 500, во втором = 1000.

#### Теория

**Уровни изоляции** определяют, насколько строго одна транзакция изолирована от изменений других транзакций, которые выполняются параллельно.

Представь, что ты читаешь книгу, а кто-то другой в это время вносит в неё правки:

*   **`READ COMMITTED` (уровень по умолчанию):** Ты читаешь страницу 1. Кто-то вносит правку в страницу 1 и сохраняет. Ты возвращаешься и читаешь страницу 1 снова, и **видишь новые данные**. Это называется **Non-repeatable Read** (неповторяющееся чтение) — один и тот же `SELECT` в рамках одной транзакции возвращает разные результаты.
*   **`REPEATABLE READ`:** Ты как будто делаешь «фотографию» (снимок/snapshot) всей книги, как только открыл её на странице 1. Сколько бы раз ты ни возвращался к странице 1, ты будешь видеть только то, что было на твоей фотографии. Изменения других людей тебе не видны до тех пор, пока ты не закроешь книгу (`COMMIT`/`ROLLBACK`).

**Наша задача:** Написать два потока. Один читает баланс, спит 1 секунду и читает снова. Второй в это время меняет баланс и сохраняет. Мы увидим, как `READ COMMITTED` пропустит изменение, а `REPEATABLE READ` — нет.

In [15]:
# УПРАЖНЕНИЕ 4

import threading
import time
from sqlalchemy import create_engine, text

# --- Создаем engine перед началом ---
SQLALCHEMY_URL = "postgresql+psycopg://postgres@localhost:5432/demo"
engine = create_engine(SQLALCHEMY_URL, echo=False)
# -----------------------------------------------

# 1. Подготовка таблицы (сбрасываем баланс на 1000)
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS ex4_accounts"))
    conn.execute(text("CREATE TABLE ex4_accounts (id INT PRIMARY KEY, balance INT)"))
    conn.execute(text("INSERT INTO ex4_accounts VALUES (1, 1000)"))

# 2. Функция для Потока-читателя (Reader)
def reader(isolation_level):
    with engine.connect().execution_options(isolation_level=isolation_level) as conn:
        with conn.begin():
            # Первое чтение
            b1 = conn.execute(text("SELECT balance FROM ex4_accounts WHERE id=1")).scalar()
            print(f"[{isolation_level}] Первое чтение: {b1}")
            
            time.sleep(1) # Ждем 1 секунду, чтобы Writer успел обновить данные
            
            # Второе чтение
            b2 = conn.execute(text("SELECT balance FROM ex4_accounts WHERE id=1")).scalar()
            print(f"[{isolation_level}] Второе чтение: {b2}")

# 3. Функция для Потока-писателя (Writer)
def writer():
    time.sleep(0.3) # Ждем 0.3 секунды, чтобы Reader успел сделать первое чтение
    with engine.begin() as conn:
        conn.execute(text("UPDATE ex4_accounts SET balance = 500 WHERE id = 1"))
        print("[Writer] Обновил баланс до 500 и сделал COMMIT")

print("--- ТЕСТ 1: READ COMMITTED ---")
# Сбрасываем баланс на 1000 перед стартом
with engine.begin() as conn:
    conn.execute(text("UPDATE ex4_accounts SET balance = 1000 WHERE id = 1"))

t_read = threading.Thread(target=reader, args=("READ COMMITTED",))
t_write = threading.Thread(target=writer)
t_read.start()
t_write.start()
t_read.join() # Ждем завершения потоков
t_write.join()

print("\n--- ТЕСТ 2: REPEATABLE READ ---")
# Снова сбрасываем баланс на 1000
with engine.begin() as conn:
    conn.execute(text("UPDATE ex4_accounts SET balance = 1000 WHERE id = 1"))

t_read = threading.Thread(target=reader, args=("REPEATABLE READ",))
t_write = threading.Thread(target=writer)
t_read.start()
t_write.start()
t_read.join()
t_write.join()

--- ТЕСТ 1: READ COMMITTED ---
[READ COMMITTED] Первое чтение: 1000
[Writer] Обновил баланс до 500 и сделал COMMIT
[READ COMMITTED] Второе чтение: 500

--- ТЕСТ 2: REPEATABLE READ ---
[REPEATABLE READ] Первое чтение: 1000
[Writer] Обновил баланс до 500 и сделал COMMIT
[REPEATABLE READ] Второе чтение: 1000


## 2.5 Best practices по транзакциям

1. **Короткие транзакции.** Пока транзакция открыта, autovacuum не может очистить старые версии строк. Возникает bloat и растёт риск wraparound.
2. **Единый порядок блокировок.** Если обновляете несколько строк/таблиц — всегда в одном и том же порядке. Иначе почти гарантированно получите deadlock (см. раздел 5).
3. **Аналитика (OLAP) — не в OLTP-базе.**
   - Идеально — вынести на streaming-реплику.
   - Если всё же на основной: `BEGIN ISOLATION LEVEL SERIALIZABLE READ ONLY DEFERRABLE;` — PG отложит старт, чтобы не конфликтовать с параллельными SERIALIZABLE.
4. **Явные блокировки — точечно.** `FOR UPDATE` превращает optimistic concurrency в pessimistic; применяйте только там, где действительно нужно (списание денег, инвентарь, бронирование).
5. **Обрабатывайте serialization failures.** На `REPEATABLE READ` / `SERIALIZABLE` приложение **обязано** уметь повторять транзакцию при ошибке.


In [16]:
# Демо: диагностика долгих транзакций
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT pid,
                   now() - xact_start AS duration,
                   state,
                   left(query, 60) AS query_head
            FROM pg_stat_activity
            WHERE xact_start IS NOT NULL
              AND backend_type = 'client backend'
              AND pid <> pg_backend_pid()
            ORDER BY duration DESC NULLS LAST
            LIMIT 10
        """)
        rows = cur.fetchall()
        print("Долгие транзакции (pid, duration, state, query):")
        for r in rows:
            print(" ", r)
        if not rows:
            print("  (нет других активных транзакций)")


Долгие транзакции (pid, duration, state, query):
  (нет других активных транзакций)


### Упражнение 5. Диагностика долгих транзакций

**Задание.** Напишите запрос, который находит все транзакции, открытые дольше 1 секунды, и показывает их `pid`, длительность, `state`, `backend_xmin` и начало запроса.

**Шаги.**
1. Напишите SQL с фильтром `now() - xact_start > interval '1 second'`.
2. Выполните его через `psycopg` — напечатайте все строки.
3. Создайте в соседнем соединении долгую транзакцию (`BEGIN REPEATABLE READ; SELECT 1;` и подержите её через `time.sleep(2)`), чтобы проверить, что запрос её находит.
4. В комментарии: какие ещё поля `pg_stat_activity` полезны для диагностики блокировок?

**Критерий успеха.** Ваш «долгий» `SELECT` из п.3 попадает в вывод.

#### Теория

Представь, что база данных начала страшно тормозить. Ты заходишь в нее как администратор и думаешь: «Кто из пользователей вешает систему?».

Для этого в PostgreSQL есть «черный ящик» — таблица **`pg_stat_activity`**. В ней видно всё, что происходит прямо сейчас.

*   **`xact_start`** — это время, когда началась транзакция.
*   Если мы вычтем из текущего времени (`now()`) время начала (`xact_start`), мы получим **длительность** транзакции (`duration`).

В нормальной системе транзакция живет миллисекунды. Если она висит **больше 1 секунды**, это подозрительно (скорее всего, кто-то забыл сделать `COMMIT`, или транзакция наткнулась на блокировку и ждет).

**Наша задача:** Научиться писать запрос, который находит «зависшие» транзакции (старше 1 секунды). Чтобы проверить его работу, мы создадим в соседнем потоке долгую транзакцию (которая поспит 2 секунды), и наш диагностический запрос должен её найти.


In [17]:
# УПРАЖНЕНИЕ 5

import time
import threading
import psycopg

def create_long_transaction():
    # Эта функция создаст "долгую" транзакцию, чтобы мы могли ее поймать
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            # Начинаем транзакцию
            cur.execute("BEGIN ISOLATION LEVEL REPEATABLE READ")
            cur.execute("SELECT 1")
            # Спим 2 секунды. В это время транзакция открыта и "висит"
            time.sleep(2) 
            cur.execute("ROLLBACK")

# Запускаем создание долгой транзакции в фоновом потоке
print("Запускаем долгую транзакцию в фоне...")
t = threading.Thread(target=create_long_transaction)
t.start()

# Ждем полсекунды, чтобы фоновая транзакция точно успела открыться
time.sleep(0.5) 

# Пишем SQL-запрос для поиска "долгих" транзакций
query = """
    SELECT pid, 
           now() - xact_start AS duration, 
           state, 
           backend_xmin, 
           left(query, 40) AS query_head
    FROM pg_stat_activity
    WHERE xact_start IS NOT NULL 
      AND now() - xact_start > interval '1 second'
      AND pid <> pg_backend_pid()
"""
# Пояснение к запросу:
# 1. xact_start IS NOT NULL - отсекаем сессии, которые вообще не открывали транзакцию.
# 2. now() - xact_start > interval '1 second' - ищем только те, что висят дольше 1 сек.
# 3. pid <> pg_backend_pid() - не показываем самих себя (наш текущий диагностический запрос).

# Выполняем диагностический запрос через psycopg
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute(query)
        rows = cur.fetchall()
        print("Найдены долгие транзакции (pid, duration, state, xmin, query):")
        for r in rows:
            print(r)

# Ждем, пока фоновый поток (с долгой транзакцией) завершится
t.join() 

# КОММЕНТАРИЙ К ЗАДАНИЮ:
# Какие еще поля pg_stat_activity полезны для диагностики блокировок?
# 1. wait_event_type и wait_event — показывают, ЧЕГО именно ждет транзакция (например, Lock, IO, Client).
# 2. state_change — время, когда транзакция перешла в текущее состояние (например, "idle in transaction").
# 3. client_addr — IP-адрес пользователя, который запустил транзакцию (чтобы понять, чей это сервис).

Запускаем долгую транзакцию в фоне...
Найдены долгие транзакции (pid, duration, state, xmin, query):


## 3. Пессимистичные блокировки и SQLAlchemy

MVCC защищает от грязного чтения, но **не спасает от Lost Update** в бизнес-логике: два пользователя прочитали одно значение, оба записали — побеждает последний.

**Пессимистичный подход** — заблокировать строку до конца транзакции через `SELECT … FOR UPDATE`.

- `SELECT … FOR UPDATE` блокирует возвращённые строки до `COMMIT`/`ROLLBACK`.
- Другие транзакции, которые тоже запросят `FOR UPDATE` (или `UPDATE`/`DELETE`) на этих строках, будут ждать.
- Обычный `SELECT` без `FOR UPDATE` **не блокируется** — читатели не страдают.
- `FOR UPDATE` дополнительно заставляет транзакцию получить XID.

**Ключевые конструкции:**
- `SET LOCAL lock_timeout = '3s'` — таймаут ожидания блокировки только до конца текущей транзакции.
- `engine.begin()` — контекстный менеджер «соединение + транзакция»: `commit()` при успехе, `rollback()` при исключении.


In [ ]:
from sqlalchemy.exc import OperationalError

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS accounts"))
    conn.execute(text("CREATE TABLE accounts (id INT PRIMARY KEY, balance INT)"))
    conn.execute(text("INSERT INTO accounts VALUES (1, 1000)"))

def withdraw_money(account_id, amount):
    try:
        with engine.begin() as conn:
            conn.execute(text("SET LOCAL lock_timeout = '3s'"))
            row = conn.execute(
                text("SELECT balance FROM accounts WHERE id = :id FOR UPDATE"),
                {"id": account_id},
            ).fetchone()
            current = row[0]
            print(f"[pid-{account_id}] баланс={current}, списываем {amount}")
            if current >= amount:
                conn.execute(
                    text("UPDATE accounts SET balance = :b WHERE id = :id"),
                    {"b": current - amount, "id": account_id},
                )
                print("  списание успешно")
            else:
                print("  недостаточно средств")
    except OperationalError as e:
        print(f"[lock timeout] {type(e).__name__}: {str(e).splitlines()[0]}")

# Одиночный вызов — проходит без ожидания
withdraw_money(1, 100)

# Проверим состояние
with engine.connect() as conn:
    print("balance =", conn.execute(text("SELECT balance FROM accounts WHERE id=1")).scalar())


### Упражнение 6. Сериализация через `FOR UPDATE`

**Задание.** Убедитесь, что два параллельных `withdraw_money` выстраиваются в очередь, а не затирают друг друга.

**Шаги.**
1. Сбросьте `accounts`: `UPDATE accounts SET balance = 1000 WHERE id = 1`.
2. Запустите два потока, каждый вызывает `withdraw_money(1, 100)`. Оба должны завершиться **успешно** (баланс 1000 ≥ 200).
3. В конце проверьте: `SELECT balance FROM accounts WHERE id = 1` — должно быть **800** (1000 − 100 − 100). Это доказывает, что второй поток дождался первого и увидел уже обновлённый баланс.
4. Дополнительно: измените `withdraw_money` так, чтобы первый вызов держал транзакцию дольше (`time.sleep(4)` после `FOR UPDATE`). Второй поток должен упасть по `lock_timeout = '3s'`.
5. В комментарии: в каком бизнес-сценарии `FOR UPDATE` необходим, а в каком — избыточен?

**Критерий успеха.** При обычном запуске двух потоков баланс = 800. При искусственной задержке — второй падает по таймауту.

#### Теория

Представь ситуацию: у тебя на счету 1000 рублей. Два человека одновременно пытаются снять по 100 рублей. 
Если база работает без блокировок (как в MVCC), оба потока прочитают баланс `1000`. Оба вычтут `100`. Оба запишут результат `900`. В итоге ты снимешь 200 рублей, а баланс всё равно будет 900. Это называется **Lost Update (Потерянное обновление)**.

Чтобы этого избежать, используют **пессимистичные блокировки**. 
Когда ты пишешь `SELECT ... FOR UPDATE`, ты говоришь базе: «Прочитай эту строку и **повесь на неё замок**». 
Пока ты не сделаешь `COMMIT` (или `ROLLBACK`), любой другой поток, который попытается прочитать эту же строку с `FOR UPDATE` (или изменить её), **зависнет и будет ждать**, пока ты не снимешь замок. 

> Это как закрыть кабинку в туалете: второй человек подойдет, дернет ручку, увидит «Занято» и будет ждать снаружи.

**Наша задача:** Запустить два потока, которые одновременно пытаются снять деньги. Благодаря `FOR UPDATE` они выстроятся в очередь: первый спишет, обновит баланс (останется 900), второй дождется своей очереди, увидит 900 и спишет еще 100 (останется 800).


In [18]:
# УПРАЖНЕНИЕ 6

import threading
import time
from sqlalchemy import text
from sqlalchemy.exc import OperationalError

# 1. Сброс баланса к исходному состоянию
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS accounts"))
    conn.execute(text("CREATE TABLE accounts (id INT PRIMARY KEY, balance INT)"))
    conn.execute(text("INSERT INTO accounts VALUES (1, 1000)"))

# 2. Функция снятия денег (с пессимистичной блокировкой)
def withdraw_money(amount, thread_name):
    try:
        with engine.begin() as conn:
            # Устанавливаем таймаут на ожидание блокировки (3 секунды).
            # Если ждать дольше - вылетит ошибка (чтобы не висеть вечно).
            conn.execute(text("SET LOCAL lock_timeout = '3s'"))
            
            # САМОЕ ВАЖНОЕ: FOR UPDATE вешает замок на эту строку до конца транзакции
            row = conn.execute(text("SELECT balance FROM accounts WHERE id=1 FOR UPDATE")).fetchone()
            current_balance = row[0]
            print(f"[{thread_name}] Прочитал баланс: {current_balance}")
            
            # Искусственная задержка (0.5 сек), чтобы второй поток наверняка подошел 
            # к этой строке и попытался захватить замок, пока первый держит его.
            time.sleep(0.5)
            
            if current_balance >= amount:
                new_balance = current_balance - amount
                # Обновляем данные
                conn.execute(text("UPDATE accounts SET balance = :b WHERE id=1"), {"b": new_balance})
                print(f"[{thread_name}] Успешно списал {amount}. Новый баланс: {new_balance}")
            else:
                print(f"[{thread_name}] Недостаточно средств")
                
    except OperationalError as e:
        print(f"[{thread_name}] ОШИБКА: Не дождался блокировки (lock_timeout)!")
        print(f"Текст ошибки: {str(e).splitlines()[0]}")

# 3. Запускаем два потока
print("Запускаем два потока на снятие по 100 рублей...")
t1 = threading.Thread(target=withdraw_money, args=(100, "Поток-1"))
t2 = threading.Thread(target=withdraw_money, args=(100, "Поток-2"))

t1.start()
t2.start()
t1.join()
t2.join()

# 4. Проверяем итоговый баланс
with engine.connect() as conn:
    final_balance = conn.execute(text("SELECT balance FROM accounts WHERE id=1")).scalar()
    print(f"\nИтоговый баланс (ожидаем 800): {final_balance}")

# КОММЕНТАРИЙ К ЗАДАНИЮ:
# В каких бизнес-сценариях нужен FOR UPDATE, а в каких избыточен?
# Нужен: Банковские переводы, продажа последнего билета/товара на складе, списание баланса бонусов.
# Избыточен: Если данные только читаются, или если конфликты происходят крайне редко 

Запускаем два потока на снятие по 100 рублей...
[Поток-2] Прочитал баланс: 1000
[Поток-2] Успешно списал 100. Новый баланс: 900
[Поток-1] Прочитал баланс: 900
[Поток-1] Успешно списал 100. Новый баланс: 800

Итоговый баланс (ожидаем 800): 800


## 4. Оптимистичный контроль версий

Альтернатива пессимистичным блокировкам — **оптимистичная блокировка (OCC)**. Не блокируем строки, а добавляем колонку `version` (или используем `xmin`). При чтении запоминаем версию; `UPDATE` требует, чтобы версия не изменилась:

```sql
UPDATE seminar_tickets
SET status = 'booked', version = version + 1
WHERE id = :id AND version = :expected;
```

Если `rowcount = 0` — кто-то уже изменил запись. Приложение **повторяет** попытку (retry).

Плюсы:
- нет блокировок → выше пропускная способность при низкой конкуренции;
- хорошо работает в веб-приложениях с короткими транзакциями.

Минусы:
- при высокой конкуренции много retry;
- нужна дисциплина: не забыть добавить `WHERE version = ...`.

**Ключевая идея:** проверка версии в `WHERE` превращает «чтение-запись» в одну атомарную операцию. Lost Update невозможен даже без `FOR UPDATE`.


In [ ]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS seminar_tickets"))
    conn.execute(text("""
        CREATE TABLE seminar_tickets (
            id INT PRIMARY KEY,
            status TEXT,
            version INT DEFAULT 0
        )
    """))
    conn.execute(text("INSERT INTO seminar_tickets (id, status) VALUES (1, 'free')"))

def book_ticket_optimistic(ticket_id, user_name, max_retries=3):
    for attempt in range(1, max_retries + 1):
        with engine.begin() as conn:
            row = conn.execute(
                text("SELECT status, version FROM seminar_tickets WHERE id = :id"),
                {"id": ticket_id},
            ).fetchone()
            if row is None:
                print(f"{user_name}: билет не существует")
                return
            status, current_version = row
            if status != 'free':
                print(f"{user_name}: билет уже занят ({status})")
                return
            res = conn.execute(
                text("""
                    UPDATE seminar_tickets
                    SET status = :s, version = version + 1
                    WHERE id = :id AND version = :v
                """),
                {"s": f"booked_by_{user_name}", "id": ticket_id, "v": current_version},
            )
            if res.rowcount == 0:
                print(f"{user_name}: конфликт версий, попытка {attempt}/{max_retries}")
                continue
            print(f"{user_name}: успешно забронировал")
            return
    print(f"{user_name}: не удалось за {max_retries} попыток")

book_ticket_optimistic(1, "alice")


### Упражнение 7. Гонка за билетом

**Задание.** Запустите 5 потоков, пытающихся одновременно забронировать один и тот же билет. Только один должен добиться успеха, остальные — увидеть «билет уже занят» (возможно, после retry).

**Шаги.**
1. Сбросьте `seminar_tickets`: `UPDATE seminar_tickets SET status='free', version=0 WHERE id=1`.
2. Создайте 5 потоков, каждый вызывает `book_ticket_optimistic(1, f"user_{i}")`.
3. Запустите все через `threading.Thread`, дождитесь `join()`.
4. Проверьте: `SELECT status, version FROM seminar_tickets WHERE id=1` — статус = `booked_by_user_X`, `version ≥ 1`.
5. Дополнительно: посчитайте, сколько успешных бронирований и сколько конфликтов.
6. В комментарии: почему оптимистичный подход даёт лучшую пропускную способность при низкой конкуренции, но хуже — при высокой?

**Критерий успеха.** Ровно 1 успешная бронь. В консоли видны `конфликт версий` и `билет уже занят` от остальных.

#### Теория

В предыдущем упражнении мы использовали «пессимистичный» подход: ставили замок (`FOR UPDATE`) и заставляли всех ждать. Это надежно, но медленно, если людей много.

**Оптимистичный подход (OCC)** работает по принципу: «Я надеюсь, что никто не будет меня обгонять, но на всякий случай проверю».

1. Мы добавляем в таблицу колонку **`version`** (версия).
2. Читаем данные (например, билет свободен, `version = 0`).
3. Когда мы хотим обновить строку, мы пишем: `UPDATE ... SET status = 'booked', version = 1 WHERE id = 1 AND version = 0`.
4. Если кто-то другой уже изменил эту строку, пока мы думали, то `version` там уже стала `1`. Наш `UPDATE` не найдет строку с `version = 0` и **обновит 0 строк**.
5. Программа видит, что обновилось 0 строк (конфликт), и либо пробует еще раз, либо говорит пользователю «Билет уже занят».

> **Аналогия:** Ты и друг одновременно редактируете Google Docs. Когда ты нажимаешь «Сохранить», система проверяет, не изменил ли файл друг. Если изменил — ты получаешь уведомление о конфликте.

**Наша задача:** Запустить 5 потоков, которые одновременно пытаются купить 1 билет. Только один должен выиграть.

In [19]:
# УПРАЖНЕНИЕ 7
import threading
from sqlalchemy import text

# Если engine не создан, раскомментируй:
# from sqlalchemy import create_engine
# engine = create_engine("postgresql+psycopg://postgres@localhost:5432/demo", echo=False)

# 1. Сброс билета (делаем его свободным, версия 0)
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS seminar_tickets"))
    conn.execute(text("""
        CREATE TABLE seminar_tickets (
            id INT PRIMARY KEY,
            status TEXT,
            version INT DEFAULT 0
        )
    """))
    conn.execute(text("INSERT INTO seminar_tickets (id, status, version) VALUES (1, 'free', 0)"))

# Счетчики для подсчета результатов (используем Lock для потокобезопасности)
success_count = 0
conflict_count = 0
lock = threading.Lock()

# 2. Функция покупки билета (оптимистичная)
def book_ticket(user_name):
    global success_count, conflict_count
    
    with engine.begin() as conn:
        # Шаг А: Читаем текущее состояние (БЕЗ блокировки FOR UPDATE)
        row = conn.execute(text("SELECT status, version FROM seminar_tickets WHERE id=1")).fetchone()
        status, current_version = row
        
        # Если билет уже кто-то купил, даже не пытаемся обновлять
        if status != 'free':
            print(f"[{user_name}] Билет уже занят ({status})")
            return
        
        # Шаг Б: Пытаемся обновить, проверяя, что версия не изменилась
        res = conn.execute(
            text("""
                UPDATE seminar_tickets 
                SET status = :s, version = version + 1 
                WHERE id = 1 AND version = :v
            """),
            {"s": f"booked_by_{user_name}", "v": current_version}
        )
        
        # Шаг В: Проверяем, сколько строк обновилось
        with lock:
            if res.rowcount > 0:
                # Если 1, значит мы успели первыми
                print(f"[{user_name}] УСПЕХ! Билет куплен.")
                success_count += 1
            else:
                # Если 0, значит кто-то другой изменил версию за наше время
                print(f"[{user_name}] КОНФЛИКТ ВЕРСИЙ! Кто-то успел раньше.")
                conflict_count += 1

# 3. Запускаем 5 потоков (5 конкурентов)
print("Запускаем гонку за 1 билет среди 5 пользователей...\n")
threads = []
for i in range(5):
    t = threading.Thread(target=book_ticket, args=(f"user_{i}",))
    threads.append(t)
    t.start()

# Ждем, пока все закончат
for t in threads:
    t.join()

# 4. Итоги
print(f"\nИтог: Успехов = {success_count}, Конфликтов = {conflict_count}")

# Проверяем финальное состояние
with engine.connect() as conn:
    final_state = conn.execute(text("SELECT status, version FROM seminar_tickets WHERE id=1")).fetchone()
    print(f"Финальное состояние билета: {final_state}")

# КОММЕНТАРИЙ К ЗАДАНИЮ:
# При низкой конкуренции (например, 1000 человек на 1000 разных билетов) оптимистичный подход 
# работает быстрее, так как нет накладных расходов на блокировки (нет очередей).
# При высокой конкуренции (как в нашем тесте, 5 человек на 1 билет) возникает много 
# конфликтов и повторных попыток (retry), что нагружает базу.

Запускаем гонку за 1 билет среди 5 пользователей...

[user_0] УСПЕХ! Билет куплен.
[user_1] КОНФЛИКТ ВЕРСИЙ! Кто-то успел раньше.
[user_2] Билет уже занят (booked_by_user_0)
[user_4] Билет уже занят (booked_by_user_0)
[user_3] Билет уже занят (booked_by_user_0)

Итог: Успехов = 1, Конфликтов = 1
Финальное состояние билета: ('booked_by_user_0', 1)


## 5. Deadlock (взаимная блокировка)

**Deadlock** — Транзакция A держит Ресурс 1 и ждёт Ресурс 2; Транзакция B держит Ресурс 2 и ждёт Ресурс 1. Цикл.

PostgreSQL обнаруживает цикл через **`deadlock_timeout`** (по умолчанию **1 секунда**): если транзакция ждёт блокировку дольше — строится граф ожиданий, ищется цикл. При обнаружении одна из транзакций откатывается с ошибкой `deadlock detected`.

Обратите внимание: `deadlock_timeout` (1 с) обычно **меньше** `lock_timeout` (например, 3 с), поэтому при deadlock вы увидите именно `DeadlockDetected`, а не `LockNotAvailable`.

**Как лечить:**
- Единый порядок захвата блокировок во всех транзакциях.
- Короткие транзакции.
- Обработка `OperationalError` с retry.

**Как диагностировать:**
- `pg_stat_activity` + `pg_blocking_pids(pid)` — кто кого держит.
- `pg_locks` — все блокировки.
- `SELECT pg_terminate_backend(<blocking_pid>)` — точечно завершить «виновника».


In [ ]:
import threading, time
from sqlalchemy.exc import OperationalError

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS accounts"))
    conn.execute(text("CREATE TABLE accounts (id INT PRIMARY KEY, balance INT)"))
    conn.execute(text("INSERT INTO accounts VALUES (1, 1000), (2, 2000)"))

def transaction_1():
    try:
        with engine.begin() as conn:
            print("T1: блокируем счёт 1")
            conn.execute(text("SELECT balance FROM accounts WHERE id = 1 FOR UPDATE"))
            time.sleep(1)
            print("T1: пытаемся заблокировать счёт 2...")
            conn.execute(text("SET LOCAL lock_timeout = '2s'"))
            conn.execute(text("SELECT balance FROM accounts WHERE id = 2 FOR UPDATE"))
            print("T1: успех")
    except OperationalError as e:
        print(f"T1 повалена: {type(e).__name__}: {str(e).splitlines()[0]}")

def transaction_2():
    try:
        with engine.begin() as conn:
            print("T2: блокируем счёт 2")
            conn.execute(text("SELECT balance FROM accounts WHERE id = 2 FOR UPDATE"))
            time.sleep(1)
            print("T2: пытаемся заблокировать счёт 1...")
            conn.execute(text("SET LOCAL lock_timeout = '2s'"))
            conn.execute(text("SELECT balance FROM accounts WHERE id = 1 FOR UPDATE"))
            print("T2: успех")
    except OperationalError as e:
        print(f"T2 повалена: {type(e).__name__}: {str(e).splitlines()[0]}")

t1 = threading.Thread(target=transaction_1)
t2 = threading.Thread(target=transaction_2)
t1.start(); t2.start()
t1.join(); t2.join()


### Упражнение 8. Устраняем deadlock

**Задание.** «Вылечите» демонстрацию: измените `transaction_2` так, чтобы она блокировала счета **в том же порядке**, что и `transaction_1` (сначала счёт 1, потом счёт 2). Deadlock должен исчезнуть.

**Шаги.**
1. Скопируйте демо-код в новую ячейку.
2. Измените `transaction_2`: вместо блокировки счёта 2 первым шагом блокируйте счёт 1.
3. Запустите скрипт **несколько раз** (3–5). Deadlock не должен появляться.
4. В комментарии объясните, почему единый порядок захвата ресурсов делает deadlock невозможным.
5. Дополнительно: напишите запрос `SELECT blocked.pid, blocking.pid, blocked.query FROM ...` с использованием `pg_blocking_pids(blocked.pid)` — выведите, кто кого блокирует, если один из потоков висит.

**Критерий успеха.** Обе транзакции всегда печатают `успех`. Ни в одном прогоне нет `DeadlockDetected`.


In [20]:
# УПРАЖНЕНИЕ 8

import threading
import time
from sqlalchemy import text
from sqlalchemy.exc import OperationalError

# 1. Подготовка данных (два счета)
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS accounts"))
    conn.execute(text("CREATE TABLE accounts (id INT PRIMARY KEY, balance INT)"))
    conn.execute(text("INSERT INTO accounts VALUES (1, 1000), (2, 2000)"))

# 2. Транзакция 1 (Блокирует Счет 1, затем Счет 2)
def transaction_1():
    try:
        with engine.begin() as conn:
            print("T1: Блокирую Счет 1...")
            conn.execute(text("SELECT balance FROM accounts WHERE id=1 FOR UPDATE"))
            time.sleep(0.5) # Пауза, чтобы T2 успела запуститься
            
            print("T1: Пытаюсь заблокировать Счет 2...")
            conn.execute(text("SELECT balance FROM accounts WHERE id=2 FOR UPDATE"))
            print("T1: УСПЕХ! Обе строки заблокированы.")
    except OperationalError as e:
        print(f"T1 повалена: {str(e).splitlines()[0]}")

# 3. Транзакция 2 (ИСПРАВЛЕНА: теперь блокирует Счет 1, затем Счет 2)
def transaction_2():
    try:
        with engine.begin() as conn:
            # ИСПРАВЛЕНИЕ: Меняем порядок. Сначала блокируем Счет 1, как и T1.
            print("T2: Блокирую Счет 1...")
            conn.execute(text("SELECT balance FROM accounts WHERE id=1 FOR UPDATE"))
            time.sleep(0.5)
            
            print("T2: Пытаюсь заблокировать Счет 2...")
            conn.execute(text("SELECT balance FROM accounts WHERE id=2 FOR UPDATE"))
            print("T2: УСПЕХ! Обе строки заблокированы.")
    except OperationalError as e:
        print(f"T2 повалена: {str(e).splitlines()[0]}")

# 4. Запуск потоков
print("Запускаем транзакции...")
t1 = threading.Thread(target=transaction_1)
t2 = threading.Thread(target=transaction_2)
t1.start()
t2.start()
t1.join()
t2.join()

print("\n--- ДИАГНОСТИКА ---")
# 5. Запрос с pg_blocking_pids для диагностики (если бы Deadlock случился, мы бы увидели кто кого держит)
# Мы просто выводим его как пример, так как сейчас транзакции уже прошли.
diagnostic_query = """
    SELECT blocked.pid AS blocked_pid, 
           blocking.pid AS blocking_pid,
           left(blocked.query, 30) AS blocked_query
    FROM pg_stat_activity blocked
    JOIN pg_stat_activity blocking 
      ON blocking.pid = ANY(pg_blocking_pids(blocked.pid))
    WHERE blocked.state = 'active';
"""
with engine.connect() as conn:
    res = conn.execute(text(diagnostic_query)).fetchall()
    print(f"Активных блокировок (ожидаем 0): {len(res)}")

# КОММЕНТАРИЙ К ЗАДАНИЮ:
# Единый порядок захвата ресурсов делает deadlock невозможным. 
# Если все транзакции захватывают строки в порядке 1 -> 2, то одна из них 
# всегда будет ждать на первом шаге, не создавая цикла.

Запускаем транзакции...
T1: Блокирую Счет 1...
T2: Блокирую Счет 1...
T1: Пытаюсь заблокировать Счет 2...
T1: УСПЕХ! Обе строки заблокированы.
T2: Пытаюсь заблокировать Счет 2...
T2: УСПЕХ! Обе строки заблокированы.

--- ДИАГНОСТИКА ---
Активных блокировок (ожидаем 0): 0
